# Challenge Telecom X - parte 2
Análisis del perfil de clientes que cancelan servicio.

Lo primero es tener los datos tratados del desafío anterior. por lo que se empezará con el código del Challenge 1.

In [5]:
# Comenzamos importando todas las librerías necesarias.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
# Nos conectamos a la fuente de datos directa:
url = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/refs/heads/main/TelecomX_Data.json"
datos = pd.read_json(url)
# Vemos las primeras filas del DataFrame.
datos.head()

,customerID,Churn,customer,phone,internet,account
0,0002-ORFBO,No,"{'gender': 'Female', 'SeniorCitizen': 0, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'One year', 'PaperlessBilling': '..."
1,0003-MKNFE,No,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'Yes'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
2,0004-TLHLJ,Yes,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
3,0011-IGKFF,Yes,"{'gender': 'Male', 'SeniorCitizen': 1, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
4,0013-EXCHZ,Yes,"{'gender': 'Female', 'SeniorCitizen': 1, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."


In [7]:
# Creamos un df por cada columna que es un JSON anidado.
customer = pd.json_normalize(datos["customer"])
phone = pd.json_normalize(datos["phone"])
internet = pd.json_normalize(datos["internet"])
account = pd.json_normalize(datos["account"])

In [8]:
# De los datos originales, solo queremos conservar la columna "Churn"
# Queremos concatenar las tablas horizontalmente, éso es el eje 1.
df = pd.concat([datos["Churn"], customer, phone, internet, account], axis=1)
df.head()

,Churn,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Charges.Monthly,Charges.Total
0,No,Female,0,Yes,Yes,9,Yes,No,DSL,No,Yes,No,Yes,Yes,No,One year,Yes,Mailed check,65.6,593.3
1,No,Male,0,No,No,9,Yes,Yes,DSL,No,No,No,No,No,Yes,Month-to-month,No,Mailed check,59.9,542.4
2,Yes,Male,0,No,No,4,Yes,No,Fiber optic,No,No,Yes,No,No,No,Month-to-month,Yes,Electronic check,73.9,280.85
3,Yes,Male,1,Yes,No,13,Yes,No,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,98.0,1237.85
4,Yes,Female,1,Yes,No,3,Yes,No,Fiber optic,No,No,No,Yes,Yes,No,Month-to-month,Yes,Mailed check,83.9,267.4


In [9]:
# Convertimos nuestros valores vacíos a NaN para poder trabajar con ellos.
df["Churn"] = df["Churn"].replace('', np.nan)

In [10]:
# Retiramos las filas con valores NaN en la columna "Churn".
df.dropna(subset=["Churn"], inplace=True)

In [11]:
# La celda pasada funcionó, por lo que podemos guardar la variable y volvemos a verificar el DataFrame.
df["Charges.Total"] = df["Charges.Total"].replace(' ', np.nan).astype("float64")

Comenzaremos con la creación de tablas de frecuencia.

In [18]:
# La información principal es saber si cancelaron o no el servicio-
frecuencia_churn = df["Churn"].value_counts()
porcentaje_churn = df["Churn"].value_counts(normalize=True).round(4)* 100
distribucion_churn = pd.DataFrame({"Frecuencia": frecuencia_churn, "Porcentaje": porcentaje_churn})
distribucion_churn.rename_axis("Churn", inplace=True)
distribucion_churn

,Frecuencia,Porcentaje
Churn,,
No,5174,73.46
Yes,1869,26.54


In [20]:
# Hacemos lo mismo para gender
frecuencia_gender = df["gender"].value_counts()
porcentaje_gender = df["gender"].value_counts(normalize=True).round(4)*100
distribucion_gender = pd.DataFrame({"Frecuencia": frecuencia_gender, "Porcentaje": porcentaje_gender})
distribucion_gender

,Frecuencia,Porcentaje
gender,,
Male,3555,50.48
Female,3488,49.52


In [23]:
# Hacemos el crosstab
frecuencia_churn_gender = pd.crosstab(df["Churn"], df["gender"])
frecuencia_churn_gender

gender,Female,Male
Churn,,
No,2549,2625
Yes,939,930


In [25]:
porcentaje_churn_gender = pd.crosstab(df["Churn"], df["gender"], normalize=True).round(4)*100
porcentaje_churn_gender

gender,Female,Male
Churn,,
No,36.19,37.27
Yes,13.33,13.20


Ahora haremos el mismo cruce para las demás variables cualitativas nominales.

In [30]:
porcentaje_churn_seniorcitizen = pd.crosstab(df["Churn"], df["SeniorCitizen"], normalize=True).round(4)*100
porcentaje_churn_seniorcitizen

SeniorCitizen,0,1
Churn,,
No,64.01,9.46
Yes,19.78,6.76


In [29]:
# Partner
porcentaje_churn_partner = pd.crosstab(df["Churn"], df["Partner"], normalize=True).round(4)*100
porcentaje_churn_partner

Partner,No,Yes
Churn,,
No,34.66,38.8
Yes,17.04,9.5


In [32]:
porcentaje_churn_dependents = pd.crosstab(df["Churn"], df["Dependents"], normalize=True).round(4)*100
porcentaje_churn_dependents

Dependents,No,Yes
Churn,,
No,48.13,25.33
Yes,21.91,4.63


In [34]:
porcentaje_churn_phone_service = pd.crosstab(df["Churn"], df["PhoneService"], normalize=True).round(4)*100
porcentaje_churn_phone_service

PhoneService,No,Yes
Churn,,
No,7.27,66.19
Yes,2.41,24.12


In [35]:
porcentaje_churn_multiple_lines = pd.crosstab(df["Churn"], df["MultipleLines"], normalize=True).round(4)*100
porcentaje_churn_multiple_lines

MultipleLines,No,No phone service,Yes
Churn,,,
No,36.08,7.27,30.12
Yes,12.05,2.41,12.07


In [36]:
porcentaje_churn_internet_service = pd.crosstab(df["Churn"], df["InternetService"], normalize=True).round(4)*100
porcentaje_churn_internet_service

InternetService,DSL,Fiber optic,No
Churn,,,
No,27.86,25.54,20.06
Yes,6.52,18.42,1.60


In [37]:
porcentaje_churn_osecurity = pd.crosstab(df["Churn"], df["OnlineSecurity"], normalize=True).round(4)*100
porcentaje_churn_osecurity

OnlineSecurity,No,No internet service,Yes
Churn,,,
No,28.92,20.06,24.48
Yes,20.74,1.60,4.19


In [38]:
porcentaje_churn_obackup = pd.crosstab(df["Churn"], df["OnlineBackup"], normalize=True).round(4)*100
porcentaje_churn_obackup

OnlineBackup,No,No internet service,Yes
Churn,,,
No,26.34,20.06,27.06
Yes,17.51,1.60,7.43


In [39]:
porcentaje_churn_dprotection = pd.crosstab(df["Churn"], df["DeviceProtection"], normalize=True).round(4)*100
porcentaje_churn_dprotection

DeviceProtection,No,No internet service,Yes
Churn,,,
No,26.75,20.06,26.65
Yes,17.19,1.60,7.74


In [40]:
porcentaje_churn_tsupport = pd.crosstab(df["Churn"], df["TechSupport"], normalize=True).round(4)*100
porcentaje_churn_tsupport

TechSupport,No,No internet service,Yes
Churn,,,
No,28.78,20.06,24.62
Yes,20.53,1.60,4.40


In [42]:
porcentaje_churn_streamingtv = pd.crosstab(df["Churn"], df["StreamingTV"], normalize=True).round(4)*100
porcentaje_churn_streamingtv

StreamingTV,No,No internet service,Yes
Churn,,,
No,26.52,20.06,26.88
Yes,13.37,1.60,11.56


In [43]:
porcentaje_churn_streamingmovies = pd.crosstab(df["Churn"], df["StreamingMovies"], normalize=True).round(4)*100
porcentaje_churn_streamingmovies

StreamingMovies,No,No internet service,Yes
Churn,,,
No,26.22,20.06,27.18
Yes,13.32,1.60,11.61


In [44]:
porcentaje_churn_contract = pd.crosstab(df["Churn"], df["Contract"], normalize=True).round(4)*100
porcentaje_churn_contract

Contract,Month-to-month,One year,Two year
Churn,,,
No,31.52,18.56,23.38
Yes,23.50,2.36,0.68


In [45]:
porcentaje_churn_peperless = pd.crosstab(df["Churn"], df["PaperlessBilling"], normalize=True).round(4)*100
porcentaje_churn_peperless

PaperlessBilling,No,Yes
Churn,,
No,34.12,39.34
Yes,6.66,19.88


In [46]:
porcentaje_churn_payment = pd.crosstab(df["Churn"], df["PaymentMethod"], normalize=True).round(4)*100
porcentaje_churn_payment

PaymentMethod,Bank transfer (automatic),Credit card (automatic),Electronic check,Mailed check
Churn,,,,
No,18.26,18.32,18.37,18.51
Yes,3.66,3.29,15.21,4.37


In [31]:
df.columns

Index(['Churn', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'Charges.Monthly', 'Charges.Total'],
      dtype='object')